In [6]:
import pandas as pd

df = pd.read_csv('api_influencers.csv')
print("Registros iniciales:", len(df))   

# 1. Limpieza estructural
nulos = df['Cuentas_Que_Sigue'].isna().sum()
df = df.dropna(subset=['Cuentas_Que_Sigue'])
print("Filas sin cuentas seguidas:", nulos)   

df['Usuario'] = df['Usuario'].str.strip().str.upper()   
duplicados = df.duplicated(subset=['Usuario']).sum()
df = df.drop_duplicates(subset=['Usuario']).reset_index(drop=True)
print("Usuarios duplicados:", duplicados)   
print("Usuarios únicos:", len(df))           

# 2. La Explosión
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.split(',')
aristas = df.explode('Cuentas_Que_Sigue')
print("Filas tras explode:", len(aristas))   

# 3. Purificación post-explosión
aristas['Cuentas_Que_Sigue'] = aristas['Cuentas_Que_Sigue'].str.strip().str.upper()
aristas = (aristas.rename(columns={'Usuario': 'Origen', 'Cuentas_Que_Sigue': 'Destino'})
                  [['Origen', 'Destino']])

self_loops = (aristas['Origen'] == aristas['Destino']).sum()
aristas = aristas[aristas['Origen'] != aristas['Destino']].reset_index(drop=True)
print("Self-loops eliminados:", self_loops)   
print("Aristas finales:", len(aristas))      
display(aristas.head(10))

Registros iniciales: 75
Filas sin cuentas seguidas: 4
Usuarios duplicados: 6
Usuarios únicos: 65
Filas tras explode: 98
Self-loops eliminados: 1
Aristas finales: 97


,Origen,Destino
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
2,FAN_6,MESSI
3,BOT_11,AMATEUR_1
4,BOT_1,AMATEUR_1
5,BOT_36,AMATEUR_1
6,FAN_4,GARNACHO
7,BOT_29,AMATEUR_1
8,BOT_13,AMATEUR_1
9,BOT_18,AMATEUR_1


## fase 2

In [2]:
import networkx as nx

G = nx.from_pandas_edgelist(aristas, source='Origen', target='Destino', create_using=nx.DiGraph())
print(f"Nodos: {G.number_of_nodes()} | Aristas: {G.number_of_edges()}")   

centralidad_entrada = pd.Series(nx.in_degree_centrality(G)).sort_values(ascending=False)
print("\nTop 3 por Centralidad de Entrada:")
display(centralidad_entrada.head(3))

Nodos: 66 | Aristas: 97

Top 3 por Centralidad de Entrada:


AMATEUR_1    0.723077
GARNACHO     0.200000
MESSI        0.153846
dtype: float64

**Pregunta: A simple vista, ¿quién parece ser el rey de la red?**
**AMATEUR_1**. Lo sigue el 72.3% de la red, más del triple que GARNACHO. Si solo contáramos seguidores, sería el patrocinio obvio.


# FASE 3

In [3]:
pagerank = pd.Series(nx.pagerank(G)).sort_values(ascending=False)
print("Top 5 por PageRank:")
display(pagerank.head(5))

Top 5 por PageRank:


AMATEUR_1    0.196941
DE_PAUL      0.088285
GARNACHO     0.084288
SUAREZ       0.079382
OTAMENDI     0.068620
dtype: float64

In [4]:

for perfil in ['AMATEUR_1', 'GARNACHO']:
    seguidores = list(G.predecessors(perfil))
    pr_seguidores = pagerank[seguidores]
    print(f"\n{perfil}")
    print(f"  Seguidores: {len(seguidores)}")
    print(f"  PageRank promedio de sus seguidores: {pr_seguidores.mean():.4f}")
    print(f"  Seguidor más influyente: {pr_seguidores.idxmax()} ({pr_seguidores.max():.4f})")
    print(f"  PageRank aportado por seguidor: {pagerank[perfil] / len(seguidores):.4f}")

print(f"\nPageRank mínimo posible en esta red: {pagerank.min():.4f}")


AMATEUR_1
  Seguidores: 47
  PageRank promedio de sus seguidores: 0.0048
  Seguidor más influyente: BOT_5 (0.0048)
  PageRank aportado por seguidor: 0.0042

GARNACHO
  Seguidores: 13
  PageRank promedio de sus seguidores: 0.0184
  Seguidor más influyente: SUAREZ (0.0794)
  PageRank aportado por seguidor: 0.0065

PageRank mínimo posible en esta red: 0.0048


**¿Quién tiene ahora la verdadera autoridad?**
PageRank todavía coloca a AMATEUR_1 en primer lugar (0.197) por volumen bruto: 47 enlaces entrantes. Pero su ventaja se reduce de 3.6× (seguidores) a 2.3× (PageRank), y la composición de ese puntaje lo delata:
- **Los 47 seguidores de AMATEUR_1 tienen el PageRank mínimo de toda la red (0.0048)**: nadie los sigue, así que transmiten la menor autoridad posible.
- **GARNACHO es seguido por la élite de la red**: SUÁREZ, NEYMAR y MESSI, que están en el top de PageRank. Sus seguidores valen en promedio 3.8 veces más, y cada uno le aporta 55% más autoridad.

La autoridad real (la que viene de nodos importantes) la tiene **GARNACHO**. El puntaje de AMATEUR_1 es cantidad sin calidad.


In [5]:
G_no_dirigido = G.to_undirected()
clustering = nx.clustering(G_no_dirigido)

print(f"Clustering AMATEUR_1: {clustering['AMATEUR_1']}")  
print(f"Clustering GARNACHO:  {clustering['GARNACHO']:.4f}")  

Clustering AMATEUR_1: 0
Clustering GARNACHO:  0.1333


## Veredicto Final

### AMATEUR_1 compró bots
- **Clustering = 0.0.** Ninguno de sus 47 seguidores está conectado con otro. La red forma una **estrella perfecta**: 47 cuentas aisladas que solo existen para seguirlo a él. Las comunidades humanas reales forman triángulos (mis seguidores se siguen entre sí); un clustering de cero con tantos seguidores es la firma topológica de una granja de bots.
- **PageRank vacío.** Su puntaje alto (0.197) viene solo del volumen. Los 47 seguidores tienen el PageRank mínimo de la red (0.0048): son cuentas que nadie sigue y que no aportan autoridad real. Cada seguidor le transmite 0.0042, contra 0.0065 de los de GARNACHO. Si se eliminan esas cuentas, AMATEUR_1 se queda sin seguidores y su PageRank colapsa al mínimo.

### GARNACHO es el influencer real
- **PageRank de calidad.** Lo siguen los nodos más autorizados de la red (SUÁREZ, NEYMAR, MESSI). El PageRank promedio de sus seguidores es 3.8 veces mayor que el de AMATEUR_1, y sin el ruido de los bots se ubica como el **2º nodo con más autoridad de toda la red** (0.145).
- **Clustering = 0.133.** Mayor que cero: sus seguidores están conectados entre sí (los fans también siguen a MESSI, y los jugadores se siguen entre ellos). Es una **comunidad orgánica**, no una estrella artificial.

**Recomendación:** Patrocinar a **GARNACHO**. AMATEUR_1 tiene más seguidores, pero cero estructura social y cero autoridad heredada. Cada peso invertido en él se gastaría en cuentas falsas.